# 04 — PyTorch LGDNet Modeling
## Credit Risk: Loss Given Default (LGD) Prediction

**Objectives:**
- Train LGDNet (PyTorch MLP) with weighted MSE tail loss
- Monitor training/validation curves and early stopping behavior
- Compare LGDNet against LightGBM and segment-average benchmark
- Inspect the tail loss weighting scheme
- Review MC Dropout uncertainty estimates

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config import load_config
from src.models.lgd_net import LGDNet, WeightedMSELoss, LGDDataset
from src.models.train import train, prepare_data

config = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Tail Loss Weighting Visualization

The weighted MSE loss assigns higher weight to predictions near LGD = 0 and LGD = 1.
This is the key modeling decision that distinguishes LGDNet from a standard regression.

In [ ]:
# Visualize weight function across LGD range
y_vals = np.linspace(0, 1, 1000)
alpha = config.model.tail_loss_alpha
weights = 1 + alpha * (np.abs(y_vals - 0.5) * 2) ** 2

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(y_vals, weights, color='#2563EB', linewidth=2)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Standard MSE (weight=1)')
ax.fill_between(y_vals, 1, weights, alpha=0.15, color='#2563EB', label='Tail emphasis region')
ax.set_xlabel('True LGD Value')
ax.set_ylabel('Loss Weight')
ax.set_title(f'WeightedMSE Loss Weight by LGD Value (alpha={alpha})')
ax.legend()
ax.grid(True, alpha=0.3)

print(f'Weight at y=0.0: {1 + alpha * (abs(0.0 - 0.5) * 2)**2:.2f}')
print(f'Weight at y=0.5: {1 + alpha * (abs(0.5 - 0.5) * 2)**2:.2f}')
print(f'Weight at y=1.0: {1 + alpha * (abs(1.0 - 0.5) * 2)**2:.2f}')

plt.tight_layout()
plt.show()

## 2. Train LGDNet

In [ ]:
# TODO: Train LGDNet
# This will log to MLflow and save model to models/lgd_model.pt

print('Training LGDNet...')
print(f'Architecture: {config.model.hidden_dims}')
print(f'Max epochs: {config.model.max_epochs}, patience: {config.model.patience}')
print(f'Batch size: {config.model.batch_size}, lr: {config.model.learning_rate}')
print()

model, scaler, metrics = train(config)

print('\n=== Training Results ===')
for split, split_metrics in metrics.items():
    print(f'\n{split}:')
    for k, v in split_metrics.items():
        print(f'  {k}: {v:.4f}')

## 3. Model Architecture Inspection

In [ ]:
print('LGDNet Architecture:')
print(model)
print()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 4. MC Dropout Uncertainty

In [ ]:
# TODO: Demonstrate MC Dropout confidence intervals
# Load test data and show distribution of CI widths

X_train, X_val, X_test, y_train, y_val, y_test, _ = prepare_data(config)

# Sample 100 test loans
n_sample = min(100, len(X_test))
X_sample = torch.tensor(X_test[:n_sample], dtype=torch.float32)

mean_pred, lower, upper = model.predict_with_uncertainty(X_sample, n_samples=100)
ci_width = (upper - lower).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ci_width, bins=30, color='#2563EB', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('90% CI Width')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of MC Dropout CI Widths')
axes[0].axvline(ci_width.mean(), color='red', linestyle='--', label=f'Mean: {ci_width.mean():.3f}')
axes[0].legend()

sorted_idx = mean_pred.numpy().argsort()
axes[1].fill_between(
    range(n_sample),
    lower.numpy()[sorted_idx],
    upper.numpy()[sorted_idx],
    alpha=0.3, color='#2563EB', label='90% CI'
)
axes[1].plot(mean_pred.numpy()[sorted_idx], color='#2563EB', linewidth=1, label='Mean prediction')
axes[1].scatter(range(n_sample), y_test[:n_sample][sorted_idx], s=5, color='red', alpha=0.5, label='Actual')
axes[1].set_xlabel('Loan (sorted by predicted LGD)')
axes[1].set_ylabel('LGD')
axes[1].set_title('Predictions with 90% CI (MC Dropout)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Mean CI width: {ci_width.mean():.4f}')
print(f'Note: MC Dropout CI is approximate epistemic uncertainty, not a calibrated interval')

## 5. LGDNet vs Benchmark Comparison

In [ ]:
# TODO: Final comparison table
# Fill in values from notebooks 03 and 04 after training

results_summary = pd.DataFrame([
    {'Model': 'Segment-Average Benchmark', 'Test MAE': None, 'Test RMSE': None, 'Test R²': None, 'MAE Improvement': 'N/A'},
    {'Model': 'Ridge Regression', 'Test MAE': None, 'Test RMSE': None, 'Test R²': None, 'MAE Improvement': 'TBD'},
    {'Model': 'LightGBM', 'Test MAE': None, 'Test RMSE': None, 'Test R²': None, 'MAE Improvement': 'TBD'},
    {'Model': 'LGDNet (PyTorch)', 'Test MAE': metrics['test']['mae'], 'Test RMSE': metrics['test']['rmse'], 'Test R²': metrics['test']['r2'], 'MAE Improvement': 'TBD'},
])

print('Model Comparison Summary (fill in baseline values from notebook 03):')
print(results_summary.to_string(index=False))

## Key Findings

*To be completed after training:*

1. **LGDNet test MAE:** [value]
2. **Improvement vs segment-average benchmark:** [%]
3. **Improvement vs LightGBM:** [%] (and conclusion about model complexity)
4. **Best epoch (early stopping):** [epoch number]
5. **MC Dropout CI:** [mean width, interpretation]
6. **Does LGDNet meet the ≥15% MAE, >0.30 R² targets?** [Yes/No with values]